### Datos (Desafíos, Estudiantes y Límites)

In [1]:
import json
with open('body.json', 'r') as file:
    body = json.load(file)
print("=== Estudiantes ===")
for estudiante in body["estudiantes"]:
    print(f"\n{estudiante['Nombre']} ({estudiante['Carrera']})")
    for index in range(len(estudiante['Postulaciones'])):
        print(f"{index + 1}°: {estudiante['Postulaciones'][index]}")

=== Estudiantes ===

Matias Ignacio Ayala Nanjari (Ingeniería Civil Telemática)
1°: SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.
2°: Clasificación de imágenes de mamografía usando Machine Learning
3°: Escalamiento de métodos de visión artificial para caracterización de biomasa, salud y proceso de alimentación de salmones.

Ignacio Andrés Araya Salinas (Ingeniería Civil Telemática)
1°: SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.
2°: Clasificación de imágenes de mamografía usando Machine Learning
3°: Escalamiento de métodos de visión artificial para caracterización de biomasa, salud y proceso de alimentación de salmones.

Camila Fernanda Guerrero Bustamante (Construcción Civil)
1°: SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.
2°: Diseño y manufactura de prototipo industrial de un equipo térmico que utilice Hidrógeno como combustible, economicamente viable para

In [2]:
print("=== Desafíos ===")
for desafio in body["desafios"]:
    print(f"\n{desafio['Titulo']}")
    for index in range(len(desafio['Carreras'])):
        print(f"{index + 1}) {desafio['Carreras'][index]}")

=== Desafíos ===

SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.
1) Arquitectura
2) Ingeniería Civil Ambiental
3) Ingeniería Civil Electrónica
4) Ingeniería Civil Informática
5) Ingeniería Civil Matemática
6) Ingeniería Civil Telemática
7) Ingeniería en Diseño de Productos

Mejorar la experiencia de entrega a cliente a Nivel Nacional
1) Ingeniería Civil
2) Ingeniería Civil Industrial
3) Ingeniería Civil Informática
4) Ingeniería Civil Telemática
5) Ingeniería Comercial
6) Ingeniería en Diseño de Productos
7) Otras

Electric Vehicle as a Service (EVaaS)
1) Ingeniería Civil Eléctrica
2) Ingeniería Civil Telemática

Mobile Water Quality Monitoring for Offshore Fish Farms
1) Ingeniería Civil Electrónica
2) Ingeniería Civil Telemática
3) Ingeniería en Diseño de Productos

Creación Modelo Revenue Management Menaje Easy
1) Ingeniería Civil
2) Ingeniería Civil Industrial
3) Ingeniería Civil Informática
4) Ingeniería Civil Matemática
5) Ingeniería Civil 

In [3]:
print("=== Límites por Carrera ===\n")
for carrera in body["carreras"]:
    print(f"{carrera['Nombre']}: {carrera['Maximo']}")

=== Límites por Carrera ===

Arquitectura: 1
Construcción Civil: 2
Ingeniería Civil: 1
Ingeniería Civil Ambiental: 1
Ingeniería Civil de Minas: 1
Ingeniería Civil Eléctrica: 1
Ingeniería Civil Electrónica: 1
Ingeniería Civil Industrial: 1
Ingeniería Civil Informática: 1
Ingeniería Civil Matemática: 1
Ingeniería Civil Mecánica: 1
Ingeniería Civil Metalúrgica: 1
Ingeniería Civil Química: 1
Ingeniería Civil Telemática: 3
Ingeniería Comercial: 1
Ingeniería en Diseño de Productos: 1
Otras: 1


### Función de Métricas

In [38]:
from collections import defaultdict
from typing import List, Dict

def print_assignments_results(assignments: List[Dict], students: List[Dict], challanges: List[Dict]):
    """
    Imprime los resultados de las asignaciones de manera organizada, incluyendo desafíos no seleccionados

    Args:
        assignments: Lista de asignaciones {Nombre, Desafio}
        students: Lista original de estudiantes
        challanges: Lista original de desafíos
    """
    # Crear un diccionario de estudiantes para acceso rápido
    students_dict = {student["Nombre"]: student for student in students}

    # Crear conjunto de todos los desafíos disponibles
    all_challanges = {challange["Titulo"] for challange in challanges}

    # Agrupar asignaciones por desafío
    assignments_by_challange = defaultdict(list)
    for assignment in assignments:
        assignments_by_challange[assignment["Desafio"]].append(assignment["Nombre"])

    # Encontrar desafíos no seleccionados
    selected_challanges = set(assignments_by_challange.keys())
    unselected_challanges = all_challanges - selected_challanges

    def get_preference_number(student: Dict, challange: str) -> int:
        """Obtiene el número de preferencia del desafío para el estudiante"""
        try:
            return student["Postulaciones"].index(challange) + 1
        except ValueError:
            return -1

    print("\n=== RESULTADOS DE ASIGNACIÓN ===")

    # Ordenar desafíos alfabéticamente
    sorted_challanges = sorted(assignments_by_challange.keys())

    for challange in sorted_challanges:
        students_in_challange = assignments_by_challange[challange]

        print(f"\nDesafío: {challange}")
        print(f"Total de estudiantes: {len(students_in_challange)}")
        print("Integrantes:")

        # Ordenar estudiantes alfabéticamente dentro de cada desafío
        for student_name in sorted(students_in_challange):
            student = students_dict[student_name]
            preference = get_preference_number(student, challange)
            pref_text = f"preferencia {preference}" if preference > 0 else "sin preferencia"
            print(f"* {student_name} ({student['Carrera']}), con {pref_text}")

    # Imprimir desafíos no seleccionados
    if unselected_challanges:
        print("\n--- DESAFÍOS NO SELECCIONADOS ---")
        for challange in sorted(unselected_challanges):
            print(f"* {challange}")

In [39]:
import numpy as np
from collections import defaultdict, Counter

def calculate_metrics(students: list, challanges: list, assignments: list, careers: list) -> dict:
    """
    Calcula métricas de calidad para una asignación de estudiantes a desafíos

    Args:
        students: Lista de estudiantes con Nombre,Carrera y Postulaciones
        challanges: Lista de desafíos con Título y Carrerras
        assignments: Lista de desafíos asignados a los estudiantes
        careers: Lista de carreras con Nombre y Máximo

    Returns:
        dict: Diccionario con todas las métricas calculadas
    """
    metrics = {}

    # Crear diccionario de equipos
    equipos = defaultdict(list)
    for assignment in assignments:
        equipos[assignment['Desafio']].append(assignment['Nombre'])

    # Diccionario para mapear estudiante a sus postulaciones y carreras
    estudiantes_postulaciones = {s['Nombre']: s['Postulaciones'] for s in students}
    estudiantes_carreras = {s['Nombre']: s['Carrera'] for s in students}

    # Diccionario de límites por carrera
    limites_carreras = {c['Nombre']: c['Maximo'] for c in careers}

    # Calcular satisfacción individual y métricas relacionadas
    satisfacciones = []
    primera_prioridad = 0
    fuera_preferencias = 0

    for assignment in assignments:
        nombre = assignment['Nombre']
        desafio = assignment['Desafio']
        postulaciones = estudiantes_postulaciones[nombre]

        try:
            indice = postulaciones.index(desafio)
            satisfaccion = 1 / (indice + 1)
            if indice == 0:
                primera_prioridad += 1
        except ValueError:
            satisfaccion = 0
            fuera_preferencias += 1

        satisfacciones.append(satisfaccion)

    # Calcular tamaños de equipos y carreras por equipo
    tamanos_equipos = []
    carreras_por_equipo = []
    equipos_por_tamano = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    equipos_exceden_limite = 0

    for desafio, miembros in equipos.items():
        if len(miembros) > 0:  # Solo considerar equipos con al menos un miembro
            tamano = len(miembros)
            tamanos_equipos.append(tamano)

            # Contar tamaños de equipo
            if tamano >= 5:
                equipos_por_tamano[5] += 1
            else:
                equipos_por_tamano[tamano] += 1

            # Contar carreras en el equipo
            carreras_equipo = Counter(estudiantes_carreras[estudiante] for estudiante in miembros)
            carreras_por_equipo.append(len(carreras_equipo))

            # Verificar límites por carrera
            excede_limite = False
            for carrera, cantidad in carreras_equipo.items():
                if cantidad > limites_carreras.get(carrera, 0):
                    excede_limite = True
                    break
            if excede_limite:
                equipos_exceden_limite += 1

    # Calcular desafíos sin equipo
    desafios_totales = set(c['Titulo'] for c in challanges)
    desafios_asignados = set(equipos.keys())
    desafios_sin_equipo = len(desafios_totales - desafios_asignados)

    # Almacenar métricas
    metrics['satisfaccion_promedio'] = np.mean(satisfacciones)
    metrics['estudiantes_primera_prioridad'] = primera_prioridad
    metrics['estudiantes_fuera_preferencias'] = fuera_preferencias
    metrics['desafios_sin_equipo'] = desafios_sin_equipo
    metrics['std_tamano_equipos'] = np.std(tamanos_equipos)
    metrics['promedio_carreras_por_equipo'] = np.mean(carreras_por_equipo)
    metrics['total_equipos'] = len([eq for eq in equipos.values() if len(eq) > 0])
    metrics['tamano_promedio_equipo'] = np.mean(tamanos_equipos)
    metrics['equipos_tamano_1'] = equipos_por_tamano[1]
    metrics['equipos_tamano_2'] = equipos_por_tamano[2]
    metrics['equipos_tamano_3'] = equipos_por_tamano[3]
    metrics['equipos_tamano_4'] = equipos_por_tamano[4]
    metrics['equipos_tamano_5_o_mas'] = equipos_por_tamano[5]
    metrics['equipos_exceden_limite_carrera'] = equipos_exceden_limite

    # Imprimir resultados
    print("\n=== Métricas de Asignación ===")
    print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
    print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
    print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
    print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
    print(f"Desviación estándar tamaño equipos: {metrics['std_tamano_equipos']:.3f}")
    print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

    print(f"\nDistribución de tamaños de equipo:")
    print(f"Equipos de 1 estudiante: {metrics['equipos_tamano_1']}")
    print(f"Equipos de 2 estudiantes: {metrics['equipos_tamano_2']}")
    print(f"Equipos de 3 estudiantes: {metrics['equipos_tamano_3']}")
    print(f"Equipos de 4 estudiantes: {metrics['equipos_tamano_4']}")
    print(f"Equipos de 5 o más estudiantes: {metrics['equipos_tamano_5_o_mas']}")

    print(f"\nMétricas de límites:")
    print(f"Equipos que exceden límites por carrera: {metrics['equipos_exceden_limite_carrera']}")

    print(f"\nMétricas adicionales:")
    print(f"Total de equipos formados: {metrics['total_equipos']}")
    print(f"Tamaño promedio de equipo: {metrics['tamano_promedio_equipo']:.2f}")

    return metrics

### Random

In [59]:
import random
from typing import List, Dict, Optional

def random_assignment(students: List[Dict], challanges: List[Dict], careers: List[Dict], assignments: Optional[List[Dict]] = None) -> List[Dict]:
    """
    Asigna aleatoriamente los estudiantes a desafíos

    Args:
        students: Lista de estudiantes con Nombre,Carrera y Postulaciones
        challanges: Lista de desafíos con Título y Carrerras
        assignments: Lista de desafíos previamente asignados a los estudiantes
        careers: Lista de carreras con Nombre y Máximo

    Returns:
        list: Lista de desafíos asignados a los estudiantes
    """
    # Inicializar resultado con asignaciones previas si existen
    result = [] if assignments is None else assignments.copy()

    # Obtener estudiantes ya asignados
    assigned_students = set() if not assignments else {a["Nombre"] for a in assignments}

    # Lista de estudiantes no asignados
    unassigned_students = [s for s in students if s["Nombre"] not in assigned_students]

    # Lista de desafíos disponibles
    available_challenges = [c["Titulo"] for c in challanges]

    # Asignar aleatoriamente los estudiantes no asignados
    for student in unassigned_students:
        challenge = random.choice(available_challenges)
        result.append({
            "Nombre": student["Nombre"],
            "Desafio": challenge
        })

    return result

assignments = random_assignment(body["estudiantes"], body["desafios"], body["carreras"])

calculate_metrics(body["estudiantes"], body["desafios"],assignments, body["carreras"])

{'satisfaccion_promedio': 0.0234375,
 'estudiantes_primera_prioridad': 0,
 'estudiantes_fuera_preferencias': 61,
 'desafios_sin_equipo': 4,
 'std_tamano_equipos': 1.0506218293818677,
 'promedio_carreras_por_equipo': 1.2941176470588236,
 'total_equipos': 34,
 'tamano_promedio_equipo': 1.8823529411764706,
 'violaciones_limites': 55}

### Heuristica

### Markov Decision Process

In [ ]:
import random
from typing import List, Dict, Optional

def mdp_assignment(students: List[Dict], challanges: List[Dict], careers: List[Dict], assignments: Optional[List[Dict]] = None) -> List[Dict]:
    """
    Asigna los estudiantes a desafíos mediante Markov Decision Process

    Args:
        students: Lista de estudiantes con Nombre,Carrera y Postulaciones
        challanges: Lista de desafíos con Título y Carrerras
        assignments: Lista de desafíos previamente asignados a los estudiantes
        careers: Lista de carreras con Nombre y Máximo

    Returns:
        list: Lista de desafíos asignados a los estudiantes
    """


#### Policy Iteration

#### Value Iteration

In [100]:
import random
from typing import List, Dict, Optional, Tuple
from collections import defaultdict
import math

def get_career_max(careers: List[Dict], career_name: str) -> int:
    """Obtiene el máximo de estudiantes permitidos para una carrera"""
    for career in careers:
        if career["Nombre"] == career_name:
            return career["Maximo"]
    return 0

def get_challenge_counts(assignments: List[Dict]) -> Dict[str, int]:
    """Obtiene el conteo de estudiantes por desafío"""
    challenge_counts = defaultdict(int)
    if assignments:
        for assignment in assignments:
            challenge_counts[assignment["Desafio"]] += 1
    return challenge_counts

def count_career_in_challenge(challenge: str, career: str, assignments: List[Dict],
                            students: List[Dict]) -> int:
    """Cuenta cuántos estudiantes de una carrera hay en un desafío"""
    return sum(1 for a in assignments
              if a["Desafio"] == challenge
              and next(s["Carrera"] for s in students
                      if s["Nombre"] == a["Nombre"]) == career)

def get_incomplete_teams(assignments: List[Dict]) -> List[str]:
    """Obtiene los desafíos con equipos incompletos (menos de 3 estudiantes)"""
    challenge_counts = get_challenge_counts(assignments)
    return [challenge for challenge, count in challenge_counts.items()
            if 0 < count < 3]

def get_challenge_score(challenge: str, student: Dict, assignments: List[Dict],
                       students: List[Dict], careers: List[Dict]) -> float:
    """
    Calcula un puntaje para un desafío basado en varios factores:
    - Prioridad en las preferencias del estudiante
    - Estado actual del equipo
    - Restricciones de carrera
    """
    score = 0.0
    challenge_counts = get_challenge_counts(assignments)
    current_count = challenge_counts[challenge]

    # Priorizar completar equipos existentes
    if 1 <= current_count < 3:
        score += 100  # Alta prioridad para completar equipos
    elif current_count == 0:
        score += 50   # Media prioridad para nuevos equipos
    elif current_count == 3:
        score += 25   # Baja prioridad para equipos casi llenos

    # Considerar preferencias del estudiante
    if challenge in student["Postulaciones"]:
        preference_index = student["Postulaciones"].index(challenge)
        score += (30 - preference_index * 10)*5  # Mayor puntaje para primeras preferencias

    # Penalizar si se excede el máximo de la carrera
    career_count = count_career_in_challenge(challenge, student["Carrera"],
                                           assignments, students)
    career_max = get_career_max(careers, student["Carrera"])
    if career_count >= career_max:
        score -= 50

    return score

def mdp_assignment(students: List[Dict], challenges: List[Dict],
                  careers: List[Dict], assignments: Optional[List[Dict]] = None) -> List[Dict]:
    """
    Asigna los estudiantes a desafíos mediante Markov Decision Process usando Policy Iteration
    con énfasis en completar equipos válidos
    """
    if assignments is None:
        assignments = []

    # Crear lista de estudiantes no asignados
    unassigned_students = [
        student for student in students
        if not any(a["Nombre"] == student["Nombre"] for a in assignments)
    ]

    # Primera fase: Asignar estudiantes priorizando completar equipos existentes
    while unassigned_students:
        # Obtener equipos incompletos
        incomplete_teams = get_incomplete_teams(assignments)

        # Seleccionar estudiante y calcular mejor asignación
        student = unassigned_students[0]
        best_score = float('-inf')
        best_challenge = None

        # Primero intentar completar equipos incompletos
        if incomplete_teams:
            for challenge in incomplete_teams:
                score = get_challenge_score(challenge, student, assignments,
                                         students, careers)
                if score > best_score:
                    best_score = score
                    best_challenge = challenge

        # Si no hay equipos incompletos o no se encontró una buena asignación,
        # intentar con todos los desafíos
        if best_challenge is None:
            for challenge in challenges:
                challenge_name = challenge["Titulo"]
                current_count = get_challenge_counts(assignments)[challenge_name]

                # Solo considerar desafíos que no estén llenos
                if current_count < 4:
                    score = get_challenge_score(challenge_name, student, assignments,
                                             students, careers)
                    if score > best_score:
                        best_score = score
                        best_challenge = challenge_name

        # Si no se encontró ninguna asignación válida, usar la primera preferencia
        # o cualquier desafío no lleno
        if best_challenge is None:
            for challenge in student["Postulaciones"]:
                if get_challenge_counts(assignments)[challenge] < 4:
                    best_challenge = challenge
                    break

            if best_challenge is None:
                for challenge in challenges:
                    if get_challenge_counts(assignments)[challenge["Titulo"]] < 4:
                        best_challenge = challenge["Titulo"]
                        break

        # Realizar la asignación
        if best_challenge:
            assignments.append({
                "Nombre": student["Nombre"],
                "Desafio": best_challenge
            })
            unassigned_students.remove(student)
        else:
            raise ValueError(f"No se pudo encontrar asignación para {student['Nombre']}")

    # Segunda fase: Balancear equipos si es necesario
    challenge_counts = get_challenge_counts(assignments)
    incomplete_teams = [c for c, count in challenge_counts.items() if 0 < count < 3]

    # Si hay equipos incompletos, intentar mover estudiantes de equipos grandes
    if incomplete_teams:
        for challenge in incomplete_teams:
            needed = 3 - challenge_counts[challenge]
            large_teams = [c for c, count in challenge_counts.items()
                         if count > 3 and c != challenge]

            if large_teams:
                for large_team in large_teams:
                    while needed > 0 and challenge_counts[large_team] > 3:
                        # Encontrar un estudiante que podamos mover
                        movable_student = next(
                            (a for a in assignments
                             if a["Desafio"] == large_team
                             and challenge in next(s["Postulaciones"]
                                                 for s in students
                                                 if s["Nombre"] == a["Nombre"])),
                            None
                        )

                        if movable_student:
                            movable_student["Desafio"] = challenge
                            challenge_counts[challenge] += 1
                            challenge_counts[large_team] -= 1
                            needed -= 1

    # Verificar estado final
    final_counts = get_challenge_counts(assignments)
    invalid_teams = [c for c, count in final_counts.items() if 0 < count < 3]

    if invalid_teams:
        # En lugar de lanzar error, intentar una última redistribución
        all_students = [a["Nombre"] for a in assignments
                       if a["Desafio"] in invalid_teams]
        other_challenges = [c for c, count in final_counts.items()
                          if count >= 3 and count < 4]

        for student_name in all_students:
            if other_challenges:
                target_challenge = random.choice(other_challenges)
                for assignment in assignments:
                    if assignment["Nombre"] == student_name:
                        assignment["Desafio"] = target_challenge
                        break

    return assignments

In [2]:
assignment = mdp_assignment(body["estudiantes"], body["desafios"], body["carreras"])
calculate_metrics(students=body["estudiantes"], challanges=body["desafios"],assignments=assignment, careers=body["carreras"])

NameError: name 'mdp_assignment' is not defined

In [47]:
print_assignments_results(assignment, body["estudiantes"], body["desafios"])


=== RESULTADOS DE ASIGNACIÓN ===

Desafío: Administrador de Diccionarios de Datos
Total de estudiantes: 3
Integrantes:
* Alexander Alfaro (Ingeniería Civil Telemática), con preferencia 1
* Cristhofer Brian Ruben Araya Recabarren (Ingeniería Civil Telemática), con preferencia 1
* Marcelo Esteban Díaz Moya (Ingeniería Civil Telemática), con sin preferencia

Desafío: Clasificación de imágenes de mamografía usando Machine Learning
Total de estudiantes: 4
Integrantes:
* Battá Tomás Tuki Cadenas (Ingeniería Civil Industrial), con preferencia 2
* Carlos Alfredo Cea Rios (Ingeniería Civil Telemática), con preferencia 1
* Ignacio Andrés Araya Salinas (Ingeniería Civil Telemática), con preferencia 2
* Matias Ignacio Ayala Nanjari (Ingeniería Civil Telemática), con preferencia 2

Desafío: Desarrollar un sistema de correlación de datos de diversas plataformas de seguridad para identificar patrones y relaciones entre eventos aparentemente no relacionados y estimar riesgos de ciberseguridad
Total d

#### Local search Iteration

### Monte Carlo Tree Search

In [75]:
import random
from typing import List, Dict, Optional
from collections import defaultdict
import math
from copy import deepcopy

class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0
        self.unassigned_students = state['unassigned']
        self.assignments = state['assignments']

    def is_terminal(self):
        return len(self.unassigned_students) == 0

    def get_possible_moves(self):
        if not self.unassigned_students:
            return []

        student = self.unassigned_students[0]
        possible_assignments = []

        # First try student's preferences
        for preference in student['Postulaciones']:
            if self._is_valid_assignment(student, preference):
                possible_assignments.append(preference)

        # If no valid preferences, try ALL other challenges
        if not possible_assignments:
            for challenge in self.state['challenges']:
                if self._is_valid_assignment(student, challenge['Titulo']):
                    possible_assignments.append(challenge['Titulo'])

        # If still no valid assignments, force assignment to any challenge
        if not possible_assignments:
            for challenge in self.state['challenges']:
                team = [a['Nombre'] for a in self.assignments if a['Desafio'] == challenge['Titulo']]
                if len(team) < 4:  # Only check team size constraint
                    possible_assignments.append(challenge['Titulo'])

        return possible_assignments

    def _is_valid_assignment(self, student, challenge_title):
        # Get current team for this challenge
        team = [a['Nombre'] for a in self.assignments if a['Desafio'] == challenge_title]

        # Check team size
        if len(team) >= 4:
            return False

        # Get career counts for current team
        career_counts = defaultdict(int)
        for member in team:
            member_career = next(s['Carrera'] for s in self.state['all_students']
                               if s['Nombre'] == member)
            career_counts[member_career] += 1

        # Add current student
        career_counts[student['Carrera']] += 1

        # Check career limits
        for career in career_counts:
            max_allowed = next(c['Maximo'] for c in self.state['careers']
                             if c['Nombre'] == career)
            if career_counts[career] > max_allowed:
                return False

        # Check if challenge accepts student's career
        challenge = next(c for c in self.state['challenges']
                        if c['Titulo'] == challenge_title)
        if student['Carrera'] not in challenge['Carreras'] and len(team) > 0:
            return False

        return True

def calculate_ucb(node, exploration_constant=1.41):
    if node.visits == 0:
        return float('inf')
    exploitation = node.value / node.visits
    exploration = exploration_constant * math.sqrt(math.log(node.parent.visits) / node.visits)
    return exploitation + exploration

def select_node(node):
    current = node
    while not current.is_terminal() and current.children:
        current = max(current.children, key=lambda c: calculate_ucb(c))
    return current

def expand_node(node):
    if node.is_terminal():
        return node

    possible_moves = node.get_possible_moves()
    if not possible_moves:
        return node

    for move in possible_moves:
        new_state = get_new_state(node.state, move)
        child = MCTSNode(new_state, parent=node)
        node.children.append(child)

    return node.children[0]

def get_new_state(state, move):
    new_state = deepcopy(state)
    student = new_state['unassigned'][0]
    new_state['unassigned'] = new_state['unassigned'][1:]
    new_state['assignments'].append({
        'Nombre': student['Nombre'],
        'Desafio': move
    })
    return new_state

def simulate(node):
    if node.is_terminal():
        return evaluate_solution(node.assignments, node.state['all_students'])

    current_state = deepcopy(node.state)
    while current_state['unassigned']:
        student = current_state['unassigned'][0]
        possible_moves = []

        # Try preferences first
        for pref in student['Postulaciones']:
            if MCTSNode(current_state)._is_valid_assignment(student, pref):
                possible_moves.append(pref)

        # Try other challenges
        if not possible_moves:
            for challenge in current_state['challenges']:
                if MCTSNode(current_state)._is_valid_assignment(student, challenge['Titulo']):
                    possible_moves.append(challenge['Titulo'])

        # If still no valid moves, force assignment to any challenge with space
        if not possible_moves:
            for challenge in current_state['challenges']:
                team = [a['Nombre'] for a in current_state['assignments']
                       if a['Desafio'] == challenge['Titulo']]
                if len(team) < 4:
                    possible_moves.append(challenge['Titulo'])

        if not possible_moves:
            return float('-inf')

        move = random.choice(possible_moves)
        current_state = get_new_state(current_state, move)

    return evaluate_solution(current_state['assignments'], current_state['all_students'])

def evaluate_solution(assignments, students):
    score = 0
    for assignment in assignments:
        student = next(s for s in students if s['Nombre'] == assignment['Nombre'])
        try:
            preference_index = student['Postulaciones'].index(assignment['Desafio'])
            score += (1/ (1 + preference_index))*4  # 4 points for 1st choice, 2 for 2nd, 1,333 for 3rd
        except ValueError:
            score += 0  # 0 points if not in preferences
    return score

def backpropagate(node, value):
    current = node
    while current:
        current.visits += 1
        current.value += value
        current = current.parent

def mcts_assignment(students: List[Dict], challenges: List[Dict], careers: List[Dict], assignments: Optional[List[Dict]] = None) -> List[Dict]:
    """
    Asigna estudiantes a desafíos usando Monte Carlo Tree Search
    """
    # Initialize state
    initial_state = {
        'unassigned': students.copy(),
        'assignments': assignments or [],
        'challenges': challenges,
        'careers': careers,
        'all_students': students.copy()
    }

    root = MCTSNode(initial_state)
    max_iterations = 10000  # Aumentado el número de iteraciones
    iteration = 0
    best_solution = None
    best_score = float('-inf')

    while True:
        iteration += 1
        leaf = select_node(root)
        if not leaf.is_terminal():
            leaf = expand_node(leaf)
            simulation_result = simulate(leaf)
            backpropagate(leaf, simulation_result)

            # Check if we have a complete valid solution
            current = root
            while current.children:
                current = max(current.children, key=lambda c: c.visits)

            if current.is_terminal():
                solution_score = evaluate_solution(current.assignments, students)
                if solution_score > best_score:
                    best_solution = current.assignments
                    best_score = solution_score

                # If we have a complete solution where all students are assigned, return it
                if len(current.assignments) == len(students):
                    return current.assignments

        # Si hemos alcanzado el máximo de iteraciones, reiniciar con nuevo árbol
        if iteration >= max_iterations:
            if best_solution and len(best_solution) == len(students):
                return best_solution
            root = MCTSNode(initial_state)
            iteration = 0

    return best_solution  # En caso de que el bucle termine por alguna razón

In [ ]:
class Node:
    def __init__(self, state, parent=None, move=None):
        self.state = deepcopy(state)
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untried_moves = state.get_legal_moves()

In [77]:
assignment = mcts_assignment(body["estudiantes"], body["desafios"], body["carreras"])
calculate_metrics(students=body["estudiantes"], challanges=body["desafios"],assignments=assignment, careers=body["carreras"])

{'satisfaccion_promedio': 0.7656249999999999,
 'estudiantes_primera_prioridad': 38,
 'estudiantes_fuera_preferencias': 0,
 'desafios_sin_equipo': 8,
 'std_tamano_equipos': 1.0873004286866728,
 'promedio_carreras_por_equipo': 1.2,
 'total_equipos': 30,
 'tamano_promedio_equipo': 2.1333333333333333,
 'violaciones_limites': 55}

In [78]:
assignment

[{'Nombre': 'Matias Ignacio Ayala Nanjari',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Ignacio Andrés Araya Salinas',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Camila Fernanda Guerrero Bustamante',
  'Desafio': 'Diseño y manufactura de prototipo industrial de un equipo térmico que utilice Hidrógeno como combustible, economicamente viable para la industria chilena.'},
 {'Nombre': 'Simón Josías Cristóbal Álvarez Díaz',
  'Desafio': 'Mobile Water Quality Monitoring for Offshore Fish Farms'},
 {'Nombre': 'Fernanda Ivonne Viera Castillo    ',
  'Desafio': 'Creación Modelo Revenue Management Menaje Easy'},
 {'Nombre': 'Manuel Cruces Pirce',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Carlos Alfredo Cea Rios',
  'Desafio': 'Clasificación de imágenes de mamografía usando M

### Approximate Dynamic Programming